# GFT-scRNA Demo: PBMC 3k

**Graph Fourier Transform** ile 2.700 kan hücresinin spektral analizi.

Bu notebook şunları gösterir:
1. Heterojenlik skoru ile GFT uygunluk testi
2. GFT embedding vs PCA karşılaştırması
3. Denoised matris ile kümeleme
4. Görselleştirme

**Gereksinimler:** `pip install -r requirements.txt`  
**Veri:** PBMC 3k (10x Genomics, 2.700 hücre, ~1 dakikada işlenir)


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from gft import build_graph, gft_embed, gft_denoise
from gft import ari_multi_seed, heterogeneity_score, predict_gft_advantage

sc.settings.verbosity = 0
SEED = 42
np.random.seed(SEED)
print("✓ Modüller yüklendi")

## 1. Veri Yükleme ve Ön İşleme

In [ ]:
# PBMC 3k — standart preprocessing
# Veri yoksa: https://cf.10xgenomics.com/samples/cell/pbmc3k/
# data/ klasörüne çıkarın

DATA_PATH = '../data/filtered_gene_bc_matrices/hg19/'

adata = sc.read_10x_mtx(DATA_PATH, var_names='gene_symbols', cache=True)
adata.var_names_make_unique()

# Standart QC + normalize
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)
sc.pp.neighbors(adata, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5, random_state=SEED)

# Bilinen PBMC etiketleri
cluster_map = {
    '0': 'CD4 T', '1': 'CD14 Mono', '2': 'CD4 T',
    '3': 'B cell', '4': 'CD8 T', '5': 'NK',
    '6': 'CD14 Mono', '7': 'DC', '8': 'CD16 Mono'
}
adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_map).fillna('Other')

le = LabelEncoder()
y  = le.fit_transform(adata.obs['cell_type'].values)
k  = len(le.classes_)
pts = adata.obsm['X_pca'][:, :50]
N   = len(pts)

print(f"✓ N={N} hücre | k={k} hücre tipi")
print(f"Hücre tipleri: {list(le.classes_)}")

## 2. GFT Uygun mu? — Heterojenlik Skoru

In [ ]:
# Veri setinin GFT için uygun olup olmadığını kontrol et
adv = predict_gft_advantage(pts, y)

print(f"Heterojenlik skoru : {adv['het_score']:.2f}")
print(f"N                  : {adv['n']}")
print(f"Öneri              : {'GFT ✓' if adv['recommended'] else 'PCA ✓'}")
print(f"Neden              : {adv['reason']}")

# Not: PBMC3k homojen bir veri seti (sadece immün hücreler, sağlıklı)
# Het < 7 → PCA önerilir. Bu demo GFT'nin sınırlarını da gösterir.

## 3. Graf İnşa ve GFT

In [ ]:
# Graf inşa — Cosine kNN
print("Graf inşa ediliyor...")
W = build_graph(pts, k_nn=20, method='cosine')
print(f"✓ Graf: {W.shape[0]} düğüm, {W.nnz//2} kenar")

# GFT embedding (k_eig=5)
print("GFT hesaplanıyor...")
emb = gft_embed(W, k_eig=5)
print(f"✓ GFT embedding: {emb.shape}")

# Denoised matris
Xh = gft_denoise(pts, emb)
print(f"✓ Denoised matris: {Xh.shape}")

## 4. Karşılaştırma: PCA vs GFT vs Denoised

In [ ]:
# ARI karşılaştırması (10 seed)
print(f"{'Yöntem':<15} | {'ARI':>8} | {'Std':>8} | {'GFT/PCA':>8}")
print("-" * 45)

pca_mean, pca_std = ari_multi_seed(pts, y, k, n_seeds=10, seed=SEED)
gft_mean, gft_std = ari_multi_seed(emb, y, k, n_seeds=10, seed=SEED)
den_mean, den_std = ari_multi_seed(Xh,  y, k, n_seeds=10, seed=SEED)

print(f"{'PCA(50)':<15} | {pca_mean:>8.4f} | {pca_std:>8.4f} | {'—':>8}")
print(f"{'GFT (k=5)':<15} | {gft_mean:>8.4f} | {gft_std:>8.4f} | {gft_mean/max(pca_mean,1e-6):>7.2f}x")
print(f"{'Denoised':<15} | {den_mean:>8.4f} | {den_std:>8.4f} | {den_mean/max(pca_mean,1e-6):>7.2f}x")
print()
if gft_mean > pca_mean:
    print(f"✓ GFT bu veri setinde PCA'yı geçti ({gft_mean/pca_mean:.2f}x)")
else:
    print(f"ℹ PCA bu veri setinde daha iyi — het_score={adv['het_score']:.1f} < 7")
    print(f"  GFT homojen veri setlerinde zayıflar. TME kanseri verisinde deneyin.")

## 5. Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0f1117')

umap = adata.obsm['X_umap']
from matplotlib.cm import get_cmap
cmap = get_cmap('tab10')
colors = [cmap(i % 10) for i in range(k)]

panels = [
    ('Gerçek Hücre Tipleri', y),
    ('GFT Kümesi (k=5)', KMeans(k, random_state=SEED, n_init=5).fit_predict(emb)),
    ('PCA Kümesi',        KMeans(k, random_state=SEED, n_init=5).fit_predict(pts)),
]

for ax, (title, labels) in zip(axes, panels):
    ax.set_facecolor('#0f1117')
    for ci in range(k):
        m = labels == ci
        lbl = le.classes_[ci] if 'Gerçek' in title else f'Küme {ci}'
        ax.scatter(umap[m,0], umap[m,1], c=[colors[ci]], s=3,
                   alpha=0.7, edgecolors='none', label=lbl)
    ari_val = adjusted_rand_score(y, labels) if 'Gerçek' not in title else 1.0
    ax.set_title(f'{title}\nARI={ari_val:.3f}', color='white', fontsize=10)
    ax.tick_params(colors='white')
    ax.set_xlabel('UMAP1', color='white')
    ax.set_ylabel('UMAP2', color='white')
    for s in ax.spines.values(): s.set_edgecolor('#333')
    if 'Gerçek' in title:
        ax.legend(fontsize=6, framealpha=0.3, labelcolor='white',
                  facecolor='#1a1a2e', markerscale=3, loc='upper right')

plt.suptitle('PBMC 3k — GFT vs PCA', color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../results/demo_pbmc3k.png', dpi=120, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("✓ Görsel kaydedildi: results/demo_pbmc3k.png")

## 6. Ne Öğrendik?

| Özellik | PBMC 3k | CMV Kohort | TME Kanser |
|---|---|---|---|
| N | 2.700 | 9.325 | 12.493 |
| k (hücre tipi) | 5 | 8 | 9 |
| Heterojenlik | ~4 (düşük) | 9.1 | 18.4 |
| GFT/PCA | ~0.7x | 2.17x | 3.21x |

**Sonuç:** PBMC3k homojen bir veri seti — GFT burada PCA'dan kötü performans gösterir.
Bu **beklenen** bir sonuç. GFT'nin asıl gücü heterojen veri setlerinde ortaya çıkar:
tümör + immün + stromal hücreler bir arada olduğunda.

Daha fazlası için:
- [experiments/01_main_comparison.py](../experiments/01_main_comparison.py) — CMV ve TME üzerinde tam analiz
- [data/README.md](../data/README.md) — heterojen veri setleri nasıl indirilir
